### Province Name Audit

In [ ]:
import pandas as pd
import numpy as np

# Diagnostic: Province Name Audit
print("Running initial province name audit...")

path_apc = "../data/processed/apc_dynasty_long.csv"
path_ira = "../data/processed/ira_funding_master_long.csv"
path_pov = "../data/processed/poverty_incidence_long.csv"

# Load the data
df_apc_raw = pd.read_csv(path_apc)
df_ira_raw = pd.read_csv(path_ira)
df_pov_raw = pd.read_csv(path_pov)

# Extract unique provinces (uppercase and stripped of spaces)
apc_provs = set(df_apc_raw['prov_name'].astype(str).str.strip().str.upper())
ira_provs = set(df_ira_raw['prov_name'].astype(str).str.strip().str.upper())
pov_provs = set(df_pov_raw['prov_name'].astype(str).str.strip().str.upper())

all_unique_provs = sorted(list(apc_provs | ira_provs | pov_provs))

print(f"Total unique provinces found across all datasets: {len(all_unique_provs)}")
print("Mismatches across datasets:")
print(f"In APC but missing elsewhere: {apc_provs - (ira_provs & pov_provs)}")
print(f"In IRA but missing elsewhere: {ira_provs - (apc_provs & pov_provs)}")
print(f"In Poverty but missing elsewhere: {pov_provs - (apc_provs & ira_provs)}\n")

### Standardization & Re-unification

In [ ]:
# The Mapping Dictionary to fix spellings and reunify splits
province_mapping = {
    'COMPOSTELA VALLEY': 'DAVAO DE ORO',
    'COTABATO': 'NORTH COTABATO',
    'MT. PROVINCE': 'MOUNTAIN PROVINCE',
    'SARANGGANI': 'SARANGANI',
    'WESTERN SAMAR': 'SAMAR',
    'SHARIFF KABUNSUAN': 'MAGUINDANAO',
    'MAGUINDANAO DEL NORTE': 'MAGUINDANAO',
    'MAGUINDANAO DEL SUR': 'MAGUINDANAO'
}

# Non-provinces to remove (NCR districts and Independent Component Cities)
drop_list = [
    '1ST DISTRICT', '2ND DISTRICT', '3RD DISTRICT', '4TH DISTRICT', 
    'COTABATO CITY', 'ISABELA CITY'
]

def process_provinces(df):
    # Standardize text
    df['prov_name'] = df['prov_name'].astype(str).str.strip().str.upper()
    
    # Map misspellings and splits
    df['prov_name'] = df['prov_name'].replace(province_mapping)
    
    # Drop non-provinces
    df = df[~df['prov_name'].isin(drop_list)]
    
    # Convert to Title Case for visual aesthetics in Tableau
    df['prov_name'] = df['prov_name'].str.title()
    
    # Aggregate data for re-unified provinces (e.g., averaging Maguindanao N & S)
    df = df.groupby(['prov_name', 'year']).mean(numeric_only=True).reset_index()
    
    return df

print("Standardizing province names and aggregating re-unified data...")
df_apc = process_provinces(df_apc_raw.copy())
df_ira = process_provinces(df_ira_raw.copy())
df_pov = process_provinces(df_pov_raw.copy())

final_prov_count = len(set(df_apc['prov_name']) | set(df_ira['prov_name']) | set(df_pov['prov_name']))
print(f"Standardization complete! Continuous province count: {final_prov_count} (Target: 81)")

### Master Temporal Grid & Merging

In [ ]:
print("Building master temporal grid...")

# Find global time boundaries
min_year = min(df_apc['year'].min(), df_ira['year'].min(), df_pov['year'].min())
max_year = max(df_apc['year'].max(), df_ira['year'].max(), df_pov['year'].max())

# Unique, cleaned provinces
all_provinces = sorted(list(set(df_apc['prov_name']) | set(df_ira['prov_name']) | set(df_pov['prov_name'])))

# Create Master Grid (Prov x Year)
grid = pd.MultiIndex.from_product(
    [all_provinces, range(min_year, max_year + 1)], 
    names=['prov_name', 'year']
)
df_master = pd.DataFrame(index=grid).reset_index()

# Merge datasets using left joins
df_master = df_master.merge(df_apc, on=['prov_name', 'year'], how='left')
df_master = df_master.merge(df_ira, on=['prov_name', 'year'], how='left')
df_master = df_master.merge(df_pov, on=['prov_name', 'year'], how='left')

print(f"Grid constructed and merged. Total rows before truncation: {len(df_master)}")

### TFT Imputation, Truncation & Feature Engineering

In [ ]:
print("Processing data: Balancing panel and backfilling parent-province records...")

# Force everything to Title Case and remove hidden spaces
df_master['prov_name'] = df_master['prov_name'].str.strip().str.title()

# Define the balanced timeline (2000-2022)
all_years = list(range(2000, 2023))
all_provinces = sorted(df_master['prov_name'].unique())

# Reindex to create the missing rows
full_index = pd.MultiIndex.from_product([all_provinces, all_years], names=['prov_name', 'year'])
df_master = df_master.set_index(['prov_name', 'year']).reindex(full_index).reset_index()

# Matching Title Case names
parent_map = {
    'Zamboanga Sibugay': ('Zamboanga Del Sur', 2001),
    'Dinagat Islands': ('Surigao Del Norte', 2006),
    'Davao Occidental': ('Davao Del Sur', 2013)
}

# Apply the copy-from-parent logic
for child, (parent, birth_year) in parent_map.items():
    if child in all_provinces and parent in all_provinces:
        parent_data = df_master[df_master['prov_name'] == parent].set_index('year')
        mask = (df_master['prov_name'] == child) & (df_master['year'] < birth_year)
        cols_to_fill = [c for c in df_master.columns if c not in ['prov_name', 'year']]
        for col in cols_to_fill:
            df_master.loc[mask, col] = df_master.loc[mask, 'year'].map(parent_data[col])

# ffill and bfill
df_master = df_master.sort_values(['prov_name', 'year'])
cols_to_fix = ['dynasty_share', 'gov_is_dynasty', 'vice_gov_is_dynasty', 'ira_funding_million_php', 'poverty_incidence']

for col in cols_to_fix:
    # First forward-fill, then back-fill for gaps at the start (year 2000)
    df_master[col] = df_master.groupby('prov_name')[col].ffill().bfill()

# Global Fill: For the few provinces with ZERO data in a specific column 
for col in cols_to_fix:
    # Fill with the yearly national median if a province has no data for that year
    df_master[col] = df_master[col].fillna(df_master.groupby('year')[col].transform('median'))

# Truncate to scope (2000-2022)
df_master = df_master[(df_master['year'] >= 2000) & (df_master['year'] <= 2022)].copy()

# Dropna should now find 0 NaNs
df_master = df_master.dropna(subset=cols_to_fix)

# Create Time Index
df_master['time_idx'] = df_master['year'] - 2000

print(f"Final Row Count: {len(df_master)}")
if len(df_master) == 1863:
    print(" SUCCESS: Perfectly balanced panel of 1863 rows achieved.")
else:
    print(f" Still missing {1863 - len(df_master)} rows. Checking for dropped provinces...")
    print(f"Remaining Provinces: {df_master['prov_name'].nunique()}")

### Final Validation & Export

In [ ]:
print("\n--- Final Dataset Validation ---")
print(f"Total Rows: {len(df_master)}")
print(f"Total Provinces represented: {df_master['prov_name'].nunique()}")
print("\nMissing Values per Column (Should all be 0):")
print(df_master.isna().sum())

# Ensure integer types for categorical flags
for col in ['gov_is_dynasty', 'vice_gov_is_dynasty']:
    if col in df_master.columns:
        df_master[col] = df_master[col].fillna(0).astype(int)

# Export
out_path = "../data/final/tft_master_dataset.csv"
df_master.to_csv(out_path, index=False)
print(f"\nModel-ready dataset successfully exported to: {out_path}")

In [ ]:
df_master.head(15)